# Practical Task 7: Batch Prediction Pipeline

## Section 1: Setup & Imports

In [1]:
# Install required library if not already installed
import subprocess
subprocess.run(['pip', 'install', 'schedule'], capture_output=True)

import sqlite3
import joblib
import numpy as np
import pandas as pd
import schedule
import time
import threading
from datetime import datetime

print('All imports successful!')
print(f'Run time: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

All imports successful!
Run time: 2026-05-03 18:25:54


## Section 2: Database Setup


In [2]:
DB_PATH = 'iris_pipeline.db'

def create_database():
    """Create the database and both required tables."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Table 1: input_data — stores flowers to predict
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS input_data (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            sepal_length    REAL NOT NULL,
            sepal_width     REAL NOT NULL,
            petal_length    REAL NOT NULL,
            petal_width     REAL NOT NULL
        )
    ''')

    cursor.execute('''
        CREATE TABLE IF NOT EXISTS predictions (
            id                   INTEGER PRIMARY KEY,
            prediction           TEXT NOT NULL,
            prediction_timestamp TEXT NOT NULL
        )
    ''')

    conn.commit()
    conn.close()
    print('Database created: iris_pipeline.db')
    print('Tables created: input_data, predictions')

create_database()

Database created: iris_pipeline.db
Tables created: input_data, predictions


## Section 3: Insert Sample Input Data


In [3]:
sample_flowers = [
    # Setosa samples
    (5.1, 3.5, 1.4, 0.2),
    (4.9, 3.0, 1.4, 0.2),
    (4.7, 3.2, 1.3, 0.2),
    (4.6, 3.1, 1.5, 0.2),
    (5.0, 3.6, 1.4, 0.2),
    (5.4, 3.9, 1.7, 0.4),
    (4.6, 3.4, 1.4, 0.3),
    (5.0, 3.4, 1.5, 0.2),
    (4.4, 2.9, 1.4, 0.2),
    (4.9, 3.1, 1.5, 0.1),
    # Versicolor samples
    (7.0, 3.2, 4.7, 1.4),
    (6.4, 3.2, 4.5, 1.5),
    (6.9, 3.1, 4.9, 1.5),
    (5.5, 2.3, 4.0, 1.3),
    (6.5, 2.8, 4.6, 1.5),
    (5.7, 2.8, 4.5, 1.3),
    (6.3, 3.3, 4.7, 1.6),
    (4.9, 2.4, 3.3, 1.0),
    (6.6, 2.9, 4.6, 1.3),
    (5.2, 2.7, 3.9, 1.4),
    # Virginica samples
    (6.3, 3.3, 6.0, 2.5),
    (5.8, 2.7, 5.1, 1.9),
    (7.1, 3.0, 5.9, 2.1),
    (6.3, 2.9, 5.6, 1.8),
    (6.5, 3.0, 5.8, 2.2),
    (7.6, 3.0, 6.6, 2.1),
    (4.9, 2.5, 4.5, 1.7),
    (7.3, 2.9, 6.3, 1.8),
    (6.7, 2.5, 5.8, 1.8),
    (7.2, 3.6, 6.1, 2.5),
]

def insert_sample_data():
    """Insert sample flowers into input_data table."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    # Only insert if table is empty (avoid duplicates on re-run)
    cursor.execute('SELECT COUNT(*) FROM input_data')
    count = cursor.fetchone()[0]

    if count == 0:
        cursor.executemany(
            'INSERT INTO input_data (sepal_length, sepal_width, petal_length, petal_width) VALUES (?, ?, ?, ?)',
            sample_flowers
        )
        conn.commit()
        print(f'Inserted {len(sample_flowers)} sample flowers into input_data')
    else:
        print(f'input_data already has {count} rows — skipping insert')

    conn.close()

insert_sample_data()

# Preview the input data
conn = sqlite3.connect(DB_PATH)
df_input = pd.read_sql('SELECT * FROM input_data', conn)
conn.close()
print(f'\nInput data preview ({len(df_input)} rows):')
df_input.head(10)

Inserted 30 sample flowers into input_data

Input data preview (30 rows):


,id,sepal_length,sepal_width,petal_length,petal_width
0,1,5.1,3.5,1.4,0.2
1,2,4.9,3.0,1.4,0.2
2,3,4.7,3.2,1.3,0.2
3,4,4.6,3.1,1.5,0.2
4,5,5.0,3.6,1.4,0.2
5,6,5.4,3.9,1.7,0.4
6,7,4.6,3.4,1.4,0.3
7,8,5.0,3.4,1.5,0.2
8,9,4.4,2.9,1.4,0.2
9,10,4.9,3.1,1.5,0.1


## Section 4: Load the Trained Model


In [5]:
SPECIES_NAMES = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}

def load_model(model_path='model.joblib'):
    """Load the trained model from disk."""
    model = joblib.load(model_path)
    print(f'Model loaded: {type(model).__name__}')
    print(f'Classes: {[SPECIES_NAMES[i] for i in model.classes_]}')
    print(f'Number of features: {model.n_features_in_}')
    return model

model = load_model()

Model loaded: LogisticRegression
Classes: ['setosa', 'versicolor', 'virginica']
Number of features: 4


## Section 5: Batch Prediction Function


In [6]:
def run_batch_prediction():
    """
    Core batch prediction function.
    Reads unprocessed input data, predicts species, stores results.
    """
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'\n{"="*50}')
    print(f'Batch prediction run started: {timestamp}')
    print(f'{"="*50}')

    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    query = '''
        SELECT i.id, i.sepal_length, i.sepal_width, i.petal_length, i.petal_width
        FROM input_data i
        LEFT JOIN predictions p ON i.id = p.id
        WHERE p.id IS NULL
    '''
    df = pd.read_sql(query, conn)

    if df.empty:
        print('No new data to predict. Pipeline is up to date.')
        conn.close()
        return

    print(f'Found {len(df)} new records to predict')

    features = df[['sepal_length', 'sepal_width',
                   'petal_length', 'petal_width']].values

    predictions = model.predict(features)
    species_names = [SPECIES_NAMES[p] for p in predictions]

    results = [
        (int(row_id), species, timestamp)
        for row_id, species in zip(df['id'], species_names)
    ]

    cursor.executemany(
        'INSERT INTO predictions (id, prediction, prediction_timestamp) VALUES (?, ?, ?)',
        results
    )
    conn.commit()

    # Show summary
    print(f'Successfully predicted {len(results)} records')
    print(f'Results saved to predictions table at {timestamp}')

    # Show prediction distribution
    from collections import Counter
    dist = Counter(species_names)
    print(f'\nPrediction breakdown:')
    for species, count in dist.items():
        print(f'  {species}: {count} flowers')

    conn.close()

print('Batch prediction function defined successfully!')

Batch prediction function defined successfully!


## Section 6: Run the First Batch


In [7]:
run_batch_prediction()


Batch prediction run started: 2026-05-03 18:27:37
Found 30 new records to predict
Successfully predicted 30 records
Results saved to predictions table at 2026-05-03 18:27:37

Prediction breakdown:
  setosa: 10 flowers
  versicolor: 11 flowers
  virginica: 9 flowers


## Section 7: View Results in the Database

In [8]:
def show_results():
    """Display the full predictions table joined with input data."""
    conn = sqlite3.connect(DB_PATH)

    query = '''
        SELECT
            i.id,
            i.sepal_length,
            i.sepal_width,
            i.petal_length,
            i.petal_width,
            p.prediction,
            p.prediction_timestamp
        FROM input_data i
        JOIN predictions p ON i.id = p.id
        ORDER BY i.id
    '''

    df = pd.read_sql(query, conn)
    conn.close()

    print(f'Total predictions stored: {len(df)}')
    print()
    return df

results_df = show_results()
results_df

Total predictions stored: 30



,id,sepal_length,sepal_width,petal_length,petal_width,prediction,prediction_timestamp
0,1,5.1,3.5,1.4,0.2,setosa,2026-05-03 18:27:37
1,2,4.9,3.0,1.4,0.2,setosa,2026-05-03 18:27:37
2,3,4.7,3.2,1.3,0.2,setosa,2026-05-03 18:27:37
3,4,4.6,3.1,1.5,0.2,setosa,2026-05-03 18:27:37
4,5,5.0,3.6,1.4,0.2,setosa,2026-05-03 18:27:37
5,6,5.4,3.9,1.7,0.4,setosa,2026-05-03 18:27:37
6,7,4.6,3.4,1.4,0.3,setosa,2026-05-03 18:27:37
7,8,5.0,3.4,1.5,0.2,setosa,2026-05-03 18:27:37
8,9,4.4,2.9,1.4,0.2,setosa,2026-05-03 18:27:37
9,10,4.9,3.1,1.5,0.1,setosa,2026-05-03 18:27:37


## Section 8: Add New Data & Run Again


In [9]:
def add_new_flowers(flowers):
    """Simulate new data arriving — insert new rows into input_data."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.executemany(
        'INSERT INTO input_data (sepal_length, sepal_width, petal_length, petal_width) VALUES (?, ?, ?, ?)',
        flowers
    )
    conn.commit()
    conn.close()
    print(f'Added {len(flowers)} new flowers to input_data')

new_batch = [
    (5.2, 3.4, 1.4, 0.2),   # should be setosa
    (6.1, 2.8, 4.7, 1.2),   # should be versicolor
    (7.2, 3.0, 5.8, 1.6),   # should be virginica
    (6.0, 2.7, 4.5, 1.4),   # borderline versicolor/virginica
    (5.0, 3.5, 1.3, 0.3),   # should be setosa
]

add_new_flowers(new_batch)
print('\nRunning pipeline on new data...')
run_batch_prediction()

Added 5 new flowers to input_data

Running pipeline on new data...

Batch prediction run started: 2026-05-03 18:28:00
Found 5 new records to predict
Successfully predicted 5 records
Results saved to predictions table at 2026-05-03 18:28:00

Prediction breakdown:
  setosa: 2 flowers
  versicolor: 2 flowers
  virginica: 1 flowers


In [10]:
results_df = show_results()
results_df.tail(10)

Total predictions stored: 35



,id,sepal_length,sepal_width,petal_length,petal_width,prediction,prediction_timestamp
25,26,7.6,3.0,6.6,2.1,virginica,2026-05-03 18:27:37
26,27,4.9,2.5,4.5,1.7,versicolor,2026-05-03 18:27:37
27,28,7.3,2.9,6.3,1.8,virginica,2026-05-03 18:27:37
28,29,6.7,2.5,5.8,1.8,virginica,2026-05-03 18:27:37
29,30,7.2,3.6,6.1,2.5,virginica,2026-05-03 18:27:37
30,31,5.2,3.4,1.4,0.2,setosa,2026-05-03 18:28:00
31,32,6.1,2.8,4.7,1.2,versicolor,2026-05-03 18:28:00
32,33,7.2,3.0,5.8,1.6,virginica,2026-05-03 18:28:00
33,34,6.0,2.7,4.5,1.4,versicolor,2026-05-03 18:28:00
34,35,5.0,3.5,1.3,0.3,setosa,2026-05-03 18:28:00


## Section 9: Scheduler


In [11]:
scheduler_running = False
scheduler_thread = None

def scheduler_loop():
    """Background loop that runs the scheduler."""
    global scheduler_running
    print('Scheduler started — pipeline will run every 5 minutes')
    print('(Running once immediately, then every 5 minutes)')
    while scheduler_running:
        schedule.run_pending()
        time.sleep(10)  # check every 10 seconds
    print('Scheduler stopped.')

def start_scheduler():
    """Start the batch prediction scheduler."""
    global scheduler_running, scheduler_thread

    if scheduler_running:
        print('Scheduler is already running!')
        return

    schedule.clear()

    schedule.every(5).minutes.do(run_batch_prediction)

    run_batch_prediction()

    scheduler_running = True
    scheduler_thread = threading.Thread(target=scheduler_loop, daemon=True)
    scheduler_thread.start()

def stop_scheduler():
    """Stop the scheduler."""
    global scheduler_running
    scheduler_running = False
    schedule.clear()
    print('Scheduler stopped.')

print('Scheduler functions defined!')
print('Run start_scheduler() to begin automatic execution')
print('Run stop_scheduler() to stop it')

Scheduler functions defined!
Run start_scheduler() to begin automatic execution
Run stop_scheduler() to stop it


In [12]:
start_scheduler()


Batch prediction run started: 2026-05-03 18:29:03
No new data to predict. Pipeline is up to date.
Scheduler started — pipeline will run every 5 minutes
(Running once immediately, then every 5 minutes)


In [14]:
print('Adding more new flowers to simulate incoming data...')

more_flowers = [
    (4.8, 3.1, 1.6, 0.2),
    (6.7, 3.0, 5.0, 1.7),
    (6.2, 2.9, 4.3, 1.3),
]

add_new_flowers(more_flowers)
print('These will be picked up in the next scheduled run (within 5 minutes)')
print('Or you can run manually:')
print('  run_batch_prediction()')

Adding more new flowers to simulate incoming data...
Added 3 new flowers to input_data
These will be picked up in the next scheduled run (within 5 minutes)
Or you can run manually:
  run_batch_prediction()


In [15]:
run_batch_prediction()


Batch prediction run started: 2026-05-03 18:29:20
Found 6 new records to predict
Successfully predicted 6 records
Results saved to predictions table at 2026-05-03 18:29:20

Prediction breakdown:
  setosa: 2 flowers
  virginica: 2 flowers
  versicolor: 2 flowers


## Section 10: Final Summary — All Predictions

In [17]:
conn = sqlite3.connect(DB_PATH)

# Summary statistics
total_input = pd.read_sql('SELECT COUNT(*) as count FROM input_data', conn).iloc[0]['count']
total_pred  = pd.read_sql('SELECT COUNT(*) as count FROM predictions', conn).iloc[0]['count']

print('='*50)
print('PIPELINE SUMMARY')
print('='*50)
print(f'Total flowers in input_data:  {int(total_input)}')
print(f'Total predictions stored:     {int(total_pred)}')
print(f'Unprocessed:                  {int(total_input - total_pred)}')
print()

dist = pd.read_sql(
    'SELECT prediction, COUNT(*) as count FROM predictions GROUP BY prediction',
    conn
)
print('Prediction distribution:')
print(dist.to_string(index=False))
print()

# Runs by timestamp (each unique timestamp = one batch run)
runs = pd.read_sql(
    'SELECT prediction_timestamp, COUNT(*) as records_processed FROM predictions GROUP BY prediction_timestamp ORDER BY prediction_timestamp',
    conn
)
print('Batch runs history:')
print(runs.to_string(index=False))

conn.close()

PIPELINE SUMMARY
Total flowers in input_data:  41
Total predictions stored:     41
Unprocessed:                  0

Prediction distribution:
prediction  count
    setosa     14
versicolor     15
 virginica     12

Batch runs history:
prediction_timestamp  records_processed
 2026-05-03 18:27:37                 30
 2026-05-03 18:28:00                  5
 2026-05-03 18:29:20                  6


In [18]:
final_df = show_results()
final_df

Total predictions stored: 41



,id,sepal_length,sepal_width,petal_length,petal_width,prediction,prediction_timestamp
0,1,5.1,3.5,1.4,0.2,setosa,2026-05-03 18:27:37
1,2,4.9,3.0,1.4,0.2,setosa,2026-05-03 18:27:37
2,3,4.7,3.2,1.3,0.2,setosa,2026-05-03 18:27:37
3,4,4.6,3.1,1.5,0.2,setosa,2026-05-03 18:27:37
4,5,5.0,3.6,1.4,0.2,setosa,2026-05-03 18:27:37
5,6,5.4,3.9,1.7,0.4,setosa,2026-05-03 18:27:37
6,7,4.6,3.4,1.4,0.3,setosa,2026-05-03 18:27:37
7,8,5.0,3.4,1.5,0.2,setosa,2026-05-03 18:27:37
8,9,4.4,2.9,1.4,0.2,setosa,2026-05-03 18:27:37
9,10,4.9,3.1,1.5,0.1,setosa,2026-05-03 18:27:37


In [19]:
stop_scheduler()
print('Pipeline demonstration complete!')

Scheduler stopped.
Pipeline demonstration complete!


Scheduler stopped.
